In [ ]:
import os
import math
import random
import time
import imageio
from collections import deque, namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import heapq

class Config:
    SEED = 0
    H = 24
    W = 24
    NUM_CATCHERS = 3
    NUM_EXITS = 3
    OBS_H = 7
    OBS_W = 7
    NUM_CELL_TYPES = 5
    GLOBAL_FEAT_DIM = 6
    MAX_EPISODE_STEPS = 1000
    NUM_MOVE_ACTIONS = 5
    NUM_BUILD_ACTIONS = 5
    ACTOR_HIDDEN = [256, 256]
    CRITIC_HIDDEN = [256, 256]
    LR_ACTOR = 3e-4
    LR_CRITIC = 1e-3
    PPO_EPOCHS = 4
    PPO_CLIP = 0.2
    GAMMA = 0.99
    GAE_LAMBDA = 0.95
    ENT_COEF = 0.01
    VALUE_COEF = 0.5
    MAX_GRAD_NORM = 0.5
    STEPS_PER_UPDATE = 2048
    MINIBATCH_SIZE = 256
    BC_EPOCHS = 100
    BC_BATCH = 256
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SAVE_DIR = "mappo_checkpoints"

torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)
random.seed(Config.SEED)

EMPTY = 0
WALL = 1
RUNNER = 2
CATCHER = 3
EXIT = 4

MOVE_DIRS = {
    0: (0, 0),
    1: (-1, 0),
    2: (1, 0),
    3: (0, -1),
    4: (0, 1),
}

class GridEnv:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.H = cfg.H
        self.W = cfg.W
        self.n = cfg.NUM_CATCHERS
        self.reset()

    def reset(self):
        self.grid = np.zeros((self.H, self.W), dtype=np.int32)
        self.grid[0,:] = WALL; self.grid[-1,:] = WALL
        self.grid[:,0] = WALL; self.grid[:,-1] = WALL

        for _ in range(int(0.06 * self.H * self.W)):
            i = random.randint(1, self.H-2)
            j = random.randint(1, self.W-2)
            self.grid[i,j] = WALL

        free = [(i,j) for i in range(1,self.H-1) for j in range(1,self.W-1) if self.grid[i,j]==EMPTY]
        random.shuffle(free)
        self.exits = []
        for k in range(self.cfg.NUM_EXITS):
            if not free: break
            e = free.pop()
            self.exits.append(e)
            self.grid[e] = EXIT

        free = [(i,j) for i in range(1,self.H-1) for j in range(1,self.W-1) if self.grid[i,j]==EMPTY]
        random.shuffle(free)
        self.runner = None
        if free:
            self.runner = free.pop()
            self.grid[self.runner] = RUNNER

        free = [(i,j) for i in range(1,self.H-1) for j in range(1,self.W-1) if self.grid[i,j]==EMPTY]
        random.shuffle(free)
        self.catchers = []
        for k in range(self.n):
            if not free: break
            p = free.pop()
            self.catchers.append(p)
            self.grid[p] = CATCHER

        self.step_count = 0
        return self.get_obs()

    def in_bounds(self, h, w):
        return 0 <= h < self.H and 0 <= w < self.W

    def get_local_window(self, pos):
        ch, cw = pos
        oh, ow = self.cfg.OBS_H, self.cfg.OBS_W
        half_h = oh // 2; half_w = ow // 2
        window = np.full((oh, ow), WALL, dtype=np.int32)
        for i in range(oh):
            for j in range(ow):
                gh = ch - half_h + i
                gw = cw - half_w + j
                if self.in_bounds(gh, gw):
                    window[i,j] = self.grid[gh, gw]
        C = self.cfg.NUM_CELL_TYPES
        onehot = np.zeros((oh, ow, C), dtype=np.float32)
        for t in range(C):
            onehot[:,:,t] = (window == t).astype(np.float32)
        return onehot

    def compute_global_features(self):
        rp = np.array([0.0, 0.0], dtype=np.float32)
        if self.runner is not None:
            rp = np.array([self.runner[0]/self.H, self.runner[1]/self.W], dtype=np.float32)
        if self.exits:
            ex = np.array(self.exits, dtype=np.float32)
            ex_mean = np.mean(ex / np.array([self.H, self.W], dtype=np.float32), axis=0)
            ex_count = len(self.exits) / max(1, self.cfg.NUM_EXITS)
        else:
            ex_mean = np.array([0.0,0.0], dtype=np.float32)
            ex_count = 0.0
        wall_density = float(np.sum(self.grid==WALL) / (self.H*self.W))
        vec = np.concatenate([rp, ex_mean, np.array([ex_count, wall_density], dtype=np.float32)])
        if getattr(self.cfg, "GLOBAL_FEAT_DIM", None) is None:
            self.cfg.GLOBAL_FEAT_DIM = len(vec)
        return vec

    def get_obs(self):
        local = [self.get_local_window(p) for p in self.catchers]
        global_feat = self.compute_global_features()
        return local, global_feat

    def update_runner(self, p_astar=0.7):
        if self.runner is None:
            return
        r_h, r_w = self.runner
        if self.runner in self.exits:
            return
        path = self.a_star_search((r_h, r_w), self.exits)
        use_astar = (len(path) >= 2) and (np.random.rand() < p_astar)
        if use_astar and len(path) >= 2:
            next_pos = path[1]
        else:
            cand = []
            for dh, dw in MOVE_DIRS.values():
                if dh == 0 and dw == 0:
                    continue
                nh, nw = r_h + dh, r_w + dw
                if not self.in_bounds(nh, nw):
                    continue
                if self.grid[nh, nw] in (EMPTY, EXIT):
                    cand.append((nh, nw))
            next_pos = random.choice(cand) if cand else (r_h, r_w)
        if next_pos != (r_h, r_w):
            if self.grid[r_h, r_w] == RUNNER:
                self.grid[r_h, r_w] = EMPTY
            nh, nw = next_pos
            self.runner = (nh, nw)
            if (nh, nw) not in self.exits:
                self.grid[nh, nw] = RUNNER

    def check_victory(self):
        if self.runner is not None and self.runner in self.exits:
            return "runner", {"reason": "runner_on_exit", "pos": self.runner}
        if self.runner is not None and self.exits:
            if not self.can_runner_reach_any_exit():
                return "catchers", {"reason": "no_path_to_exit"}
        return None, {}

    def can_runner_reach_any_exit(self):
        if self.runner is None or not self.exits:
            return False
        for exit_pos in self.exits:
            path = self.a_star_search(self.runner, [exit_pos])
            if path:
                return True
        return False

    def a_star_search(self, start, goals):
        if not goals:
            return []
        start = tuple(start)
        goals = [tuple(g) for g in goals]
        if start in goals:
            return [start]
        def heuristic(a, b):
            return abs(a[0]-b[0]) + abs(a[1]-b[1])
        closest_goal = min(goals, key=lambda g: heuristic(start, g))
        open_set = []
        heapq.heappush(open_set, (heuristic(start, closest_goal), 0, start))
        came_from = {}
        g_score = {start: 0}
        f_score = {start: heuristic(start, closest_goal)}
        while open_set:
            current_f, current_g, current = heapq.heappop(open_set)
            if current in goals:
                path = [current]
                while current in came_from:
                    current = came_from[current]
                    path.append(current)
                return list(reversed(path))
            for dh, dw in MOVE_DIRS.values():
                if dh == 0 and dw == 0:
                    continue
                neighbor = (current[0] + dh, current[1] + dw)
                if not self.in_bounds(*neighbor):
                    continue
                cell_type = self.grid[neighbor]
                if cell_type == WALL:
                    continue
                tentative_g_score = g_score[current] + 1
                if neighbor not in g_score or tentative_g_score < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g_score
                    closest_goal = min(goals, key=lambda g: heuristic(neighbor, g))
                    f_score[neighbor] = tentative_g_score + heuristic(neighbor, closest_goal)
                    heapq.heappush(open_set, (f_score[neighbor], tentative_g_score, neighbor))
        return []

    def step(self, actions_move, actions_build, runner_random_prob=0):
        self.update_runner(p_astar=runner_random_prob)
        desired = []
        for idx, mv in enumerate(actions_move):
            ch, cw = self.catchers[idx]
            dh, dw = MOVE_DIRS.get(int(mv), (0,0))
            nh, nw = ch + dh, cw + dw
            if not self.in_bounds(nh, nw) or self.grid[nh,nw] in (WALL, EXIT, RUNNER):
                desired.append((ch,cw))
            else:
                desired.append((nh,nw))
        counts = {}
        for pos in desired:
            counts[pos] = counts.get(pos, 0) + 1
        final = []
        for idx, pos in enumerate(desired):
            if counts[pos] > 1:
                final.append(self.catchers[idx])
            else:
                final.append(pos)
        for (h,w) in self.catchers:
            if self.grid[h,w] == CATCHER:
                self.grid[h,w] = EMPTY
        self.catchers = final
        for (h,w) in self.catchers:
            if self.grid[h,w] == EMPTY:
                self.grid[h,w] = CATCHER

        rewards = np.zeros((self.n,), dtype=np.float32)

        for idx, bd in enumerate(actions_build):
            if bd == 0:
                continue
            ch, cw = self.catchers[idx]
            dh, dw = MOVE_DIRS.get(int(bd), (0,0))
            bh, bw = ch + dh, cw + dw
            if self.in_bounds(bh, bw) and self.grid[bh, bw] == EMPTY:
                self.grid[bh, bw] = WALL
                for ex in self.exits:
                    if abs(bh - ex[0]) + abs(bw - ex[1]) == 1:
                        rewards[:] = 1.0
                        break

        self.step_count += 1
        winner, info = self.check_victory()
        done = False
        if winner == "runner":
            rewards[:] = -5.0
            done = True
        elif winner == "catchers":
            rewards[:] = 5.0
            done = True
        elif self.step_count >= self.cfg.MAX_EPISODE_STEPS:
            done = True
        obs = self.get_obs()
        return obs, rewards, done, {"winner": winner, **info}

    def render_frame(self, scale=8):
        H, W = self.H, self.W
        img = np.ones((H*scale, W*scale, 3), dtype=np.uint8) * 240
        color = {
            EMPTY: (240,240,240),
            WALL: (60,60,60),
            RUNNER: (220,60,60),
            CATCHER: (60,120,220),
            EXIT: (250,210,0)
        }
        for i in range(H):
            for j in range(W):
                c = self.grid[i,j]
                img[i*scale:(i+1)*scale, j*scale:(j+1)*scale] = color[c]
        return img

def mlp(input_dim, hidden_sizes, output_dim, activation=nn.ReLU):
    layers = []
    prev = input_dim
    for h in hidden_sizes:
        layers.append(nn.Linear(prev, h)); layers.append(activation()); prev = h
    layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)

class ActorNet(nn.Module):
    def __init__(self, obs_h, obs_w, n_types, move_dim, build_dim, hidden_sizes):
        super().__init__()
        self.obs_dim = obs_h * obs_w * n_types
        self.move_dim = move_dim
        self.build_dim = build_dim
        self.conv = nn.Sequential(
            nn.Conv2d(n_types, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        conv_output_size = 64 * obs_h * obs_w
        self.net = mlp(conv_output_size, hidden_sizes, self.move_dim + self.build_dim, activation=nn.ReLU)

    def forward(self, obs):
        B = obs.shape[0]
        x = obs.permute(0, 3, 1, 2)
        x = self.conv(x)
        logits = self.net(x)
        move_logits = logits[:, :self.move_dim]
        build_logits = logits[:, self.move_dim:]
        return move_logits, build_logits

class CriticNet(nn.Module):
    def __init__(self, global_dim, hidden_sizes):
        super().__init__()
        self.net = mlp(global_dim, hidden_sizes, 1, activation=nn.ReLU)

    def forward(self, g):
        return self.net(g).squeeze(-1)

class MAPPO:
    def __init__(self, cfg: Config, share_actor=False):
        self.cfg = cfg
        self.device = cfg.DEVICE
        self.share_actor = share_actor
        if share_actor:
            self.actor = ActorNet(cfg.OBS_H, cfg.OBS_W, cfg.NUM_CELL_TYPES,
                                  cfg.NUM_MOVE_ACTIONS, cfg.NUM_BUILD_ACTIONS, cfg.ACTOR_HIDDEN).to(self.device)
            self.actors = [self.actor] * cfg.NUM_CATCHERS
        else:
            self.actors = [ActorNet(cfg.OBS_H, cfg.OBS_W, cfg.NUM_CELL_TYPES,
                                    cfg.NUM_MOVE_ACTIONS, cfg.NUM_BUILD_ACTIONS, cfg.ACTOR_HIDDEN).to(self.device)
                           for _ in range(cfg.NUM_CATCHERS)]
        tmp_env = GridEnv(cfg)
        G = tmp_env.compute_global_features().shape[0]
        cfg.GLOBAL_FEAT_DIM = G
        self.critic = CriticNet(G, cfg.CRITIC_HIDDEN).to(self.device)
        actor_params = []
        if share_actor:
            actor_params = list(self.actor.parameters())
        else:
            for a in self.actors:
                actor_params += list(a.parameters())
        self.opt_actor = optim.Adam(actor_params, lr=cfg.LR_ACTOR)
        self.opt_critic = optim.Adam(self.critic.parameters(), lr=cfg.LR_CRITIC)

    def act(self, obs_per_agent):
        if isinstance(obs_per_agent, list):
            obs_arr = np.stack(obs_per_agent, axis=0)
        else:
            obs_arr = obs_per_agent
        n = obs_arr.shape[0]
        move_actions = []
        build_actions = []
        logps = []
        for i in range(n):
            ob = torch.tensor(obs_arr[i:i+1], dtype=torch.float32, device=self.device)
            actor = self.actors[i] if not self.share_actor else self.actor
            with torch.no_grad():
                move_logits, build_logits = actor(ob)
                move_dist = torch.distributions.Categorical(logits=move_logits)
                build_dist = torch.distributions.Categorical(logits=build_logits)
                ma = move_dist.sample().cpu().item()
                ba = build_dist.sample().cpu().item()
                lp = (move_dist.log_prob(torch.tensor(ma, device=self.device)) + build_dist.log_prob(torch.tensor(ba, device=self.device))).cpu().item()
            move_actions.append(int(ma))
            build_actions.append(int(ba))
            logps.append(lp)
        return np.array(move_actions, dtype=np.int32), np.array(build_actions, dtype=np.int32), np.array(logps, dtype=np.float32)

    def evaluate_actions(self, obs_batch, move_actions, build_actions):
        ob = torch.tensor(obs_batch, dtype=torch.float32, device=self.device)
        move_logits, build_logits = self.actor(ob) if self.share_actor else self._eval_multi_actor(ob)
        move_dist = torch.distributions.Categorical(logits=move_logits)
        build_dist = torch.distributions.Categorical(logits=build_logits)
        mv = torch.tensor(move_actions, dtype=torch.long, device=self.device)
        bd = torch.tensor(build_actions, dtype=torch.long, device=self.device)
        logp = move_dist.log_prob(mv) + build_dist.log_prob(bd)
        entropy = move_dist.entropy().mean() + build_dist.entropy().mean()
        return logp, entropy

    def _eval_multi_actor(self, ob):
        return self.actors[0](ob)

    def value(self, global_batch):
        g = torch.tensor(global_batch, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            v = self.critic(g).cpu().numpy()
        return v

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        torch.save({
            "actor_state": self.actor.state_dict() if self.share_actor else [a.state_dict() for a in self.actors],
            "critic_state": self.critic.state_dict()
        }, os.path.join(path, "mappo.pth"))

    def load(self, path):
        ck = torch.load(os.path.join(path, "mappo.pth"), map_location=self.device)
        if self.share_actor:
            self.actor.load_state_dict(ck["actor_state"])
        else:
            states = ck["actor_state"]
            for a, s in zip(self.actors, states):
                a.load_state_dict(s)
        self.critic.load_state_dict(ck["critic_state"])

def compute_gae(rewards, values, dones, gamma, lam):
    T = len(rewards)
    adv = np.zeros_like(rewards, dtype=np.float32)
    lastgaelam = 0
    for t in reversed(range(T)):
        nonterminal = 1.0 - float(dones[t])
        nextval = values[t+1] if t+1 < len(values) else 0.0
        delta = rewards[t] + gamma * nextval * nonterminal - values[t]
        lastgaelam = delta + gamma * lam * nonterminal * lastgaelam
        adv[t] = lastgaelam
    returns = adv + values[:T]
    return adv, returns

def load_demos_numeric(demo_path):
    data = np.load(demo_path, allow_pickle=True)
    keys = ["obs_local", "actions_move", "actions_build", "rewards", "done", "global_feat"]
    for k in ["obs_local", "actions_move", "actions_build"]:
        if k not in data:
            raise KeyError(f"В {demo_path} нет ключа '{k}'")
    obs_raw = data["obs_local"]
    moves_raw = data["actions_move"]
    builds_raw = data["actions_build"]
    rewards_raw = data.get("rewards", None)
    dones_raw = data.get("done", None)
    if isinstance(obs_raw, np.ndarray) and obs_raw.dtype != object and obs_raw.ndim == 5:
        obs_arr = obs_raw.astype(np.float32)
    else:
        try:
            T = len(obs_raw)
            first = obs_raw[0]
            n_agents = len(first)
            sample = np.array(first[0], dtype=np.float32)
            H, W, C = sample.shape
            obs_arr = np.zeros((T, n_agents, H, W, C), dtype=np.float32)
            for t in range(T):
                step = obs_raw[t]
                if len(step) != n_agents:
                    raise ValueError(f"Step {t} has {len(step)} agents, expected {n_agents}")
                for a in range(n_agents):
                    arr = np.array(step[a], dtype=np.float32)
                    if arr.shape != (H, W, C):
                        raise ValueError(f"Agent {a} at step {t} has shape {arr.shape}, expected {(H,W,C)}")
                    obs_arr[t, a] = arr
        except Exception as e:
            raise ValueError("Не удалось привести obs_local к числовому массиву: " + str(e))
    def to_int2(x, name):
        if isinstance(x, np.ndarray) and x.dtype != object and x.ndim == 2:
            return x.astype(np.int32)
        if isinstance(x, np.ndarray) and x.dtype == object:
            try:
                return np.stack([np.array(step, dtype=np.int32) for step in x], axis=0)
            except Exception as e:
                raise ValueError(f"Не удалось преобразовать {name}: {e}")
        arr = np.array(x, dtype=np.int32)
        if arr.ndim == 2:
            return arr
        raise ValueError(f"Формат {name} не распознан")
    moves = to_int2(moves_raw, "actions_move")
    builds = to_int2(builds_raw, "actions_build")
    rewards = None
    dones = None
    if rewards_raw is not None:
        rewards = np.array(rewards_raw, dtype=np.float32)
    if dones_raw is not None:
        dones = np.array(dones_raw, dtype=np.bool_)
    T1, n1, H1, W1, C1 = obs_arr.shape
    T2, n2 = moves.shape
    T3, n3 = builds.shape
    if not (T1 == T2 == T3 and n1 == n2 == n3):
        raise ValueError(f"Несоответствие форм: obs T={T1},moves T={T2},builds T={T3}; agents obs n={n1},moves n={n2},builds n={n3}")
    return obs_arr, moves, builds, rewards, dones

def pretrain_bc(mappo, demo_path="demo_data.npz", epochs=100, batch_size=256):
    obs_local, moves, builds, rewards, dones = load_demos_numeric(demo_path)
    T, n, H, W, C = obs_local.shape
    B = T * n
    obs_flat = obs_local.reshape(B, H, W, C)
    mv_flat = moves.reshape(B)
    bd_flat = builds.reshape(B)
    device = mappo.device
    actor = mappo.actor if mappo.share_actor else mappo.actors[0]
    opt = mappo.opt_actor
    criterion = nn.CrossEntropyLoss()
    idxs = np.arange(B)
    for ep in range(epochs):
        np.random.shuffle(idxs)
        losses = []
        for i in range(0, B, batch_size):
            batch = idxs[i:i+batch_size]
            ob = torch.tensor(obs_flat[batch], dtype=torch.float32, device=device)
            mv = torch.tensor(mv_flat[batch], dtype=torch.long, device=device)
            bd = torch.tensor(bd_flat[batch], dtype=torch.long, device=device)
            opt.zero_grad()
            move_logits, build_logits = actor(ob)
            loss = criterion(move_logits, mv) + criterion(build_logits, bd)
            loss.backward()
            nn.utils.clip_grad_norm_(actor.parameters(), mappo.cfg.MAX_GRAD_NORM if hasattr(mappo.cfg, "MAX_GRAD_NORM") else 0.5)
            opt.step()
            losses.append(loss.item())
        print(f"[BC] epoch {ep+1}/{epochs}, loss={np.mean(losses):.6f}")
    return mappo

def collect_rollout(env: GridEnv, mappo: MAPPO, steps_per_update, eval_gif_every=10):
    obs_local, global_feat = env.reset()
    traj_obs, traj_global, traj_move, traj_build, traj_reward, traj_done, traj_logp = ([] for _ in range(7))
    total_env_steps = 0
    episode_counter = 0
    episode_summaries = []
    current_episode_frames = []
    current_episode_steps = 0

    while total_env_steps < steps_per_update:
        obs_arr = np.stack(obs_local, axis=0)
        mv, bd, logp = mappo.act(obs_arr)
        obs_tuple, rewards, done, info = env.step(list(mv), list(bd))
        winner = info.get('winner', None)
        for a in range(len(mv)):
            traj_obs.append(obs_arr[a])
            traj_global.append(global_feat)
            traj_move.append(int(mv[a]))
            traj_build.append(int(bd[a]))
            traj_reward.append(float(rewards[a]))
            traj_done.append(bool(done))
            traj_logp.append(float(logp[a]))
        current_episode_frames.append(env.render_frame())
        current_episode_steps += 1
        if isinstance(obs_tuple, tuple) and len(obs_tuple) == 2:
            obs_local, global_feat = obs_tuple
        else:
            obs_local = obs_tuple
            global_feat = env.compute_global_features()
        total_env_steps += len(mv)
        if done:
            episode_counter += 1
            actual_winner = info.get('winner', 'unknown')
            episode_summaries.append({
                "episode_idx": episode_counter,
                "winner": actual_winner,
                "steps": current_episode_steps,
                "frames": current_episode_frames.copy()
            })
            current_episode_frames = []
            current_episode_steps = 0
            obs_local, global_feat = env.reset()

    obs_flat = np.stack(traj_obs, axis=0)
    global_flat = np.stack(traj_global, axis=0)
    move_flat = np.array(traj_move, dtype=np.int32)
    build_flat = np.array(traj_build, dtype=np.int32)
    reward_flat = np.array(traj_reward, dtype=np.float32)
    done_flat = np.array(traj_done, dtype=np.bool_)
    old_logp_flat = np.array(traj_logp, dtype=np.float32)
    return obs_flat, global_flat, move_flat, build_flat, reward_flat, done_flat, old_logp_flat, episode_summaries

def ppo_update(mappo: MAPPO, obs_flat, global_flat, move_flat, build_flat, reward_flat, done_flat, old_logp_flat, cfg: Config):
    device = mappo.device
    B = obs_flat.shape[0]
    with torch.no_grad():
        values = mappo.critic(torch.tensor(global_flat, dtype=torch.float32, device=device)).cpu().numpy()
    values_full = np.concatenate([values, np.array([0.0])], axis=0)
    adv, returns = compute_gae(reward_flat, values_full, done_flat, cfg.GAMMA, cfg.GAE_LAMBDA)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    obs_t = torch.tensor(obs_flat, dtype=torch.float32, device=device)
    global_t = torch.tensor(global_flat, dtype=torch.float32, device=device)
    mv_t = torch.tensor(move_flat, dtype=torch.long, device=device)
    bd_t = torch.tensor(build_flat, dtype=torch.long, device=device)
    old_logp_t = torch.tensor(old_logp_flat, dtype=torch.float32, device=device)
    adv_t = torch.tensor(adv, dtype=torch.float32, device=device)
    ret_t = torch.tensor(returns, dtype=torch.float32, device=device)
    idxs = np.arange(B)
    for epoch in range(cfg.PPO_EPOCHS):
        np.random.shuffle(idxs)
        for start in range(0, B, cfg.MINIBATCH_SIZE):
            mb_idx = idxs[start:start+cfg.MINIBATCH_SIZE]
            mb_obs = obs_t[mb_idx]
            mb_g = global_t[mb_idx]
            mb_mv = mv_t[mb_idx]
            mb_bd = bd_t[mb_idx]
            mb_old_logp = old_logp_t[mb_idx]
            mb_adv = adv_t[mb_idx]
            mb_ret = ret_t[mb_idx]
            if mappo.share_actor:
                move_logits, build_logits = mappo.actor(mb_obs)
            else:
                move_logits, build_logits = mappo.actors[0](mb_obs)
            move_dist = torch.distributions.Categorical(logits=move_logits)
            build_dist = torch.distributions.Categorical(logits=build_logits)
            logp = move_dist.log_prob(mb_mv) + build_dist.log_prob(mb_bd)
            entropy = move_dist.entropy().mean() + build_dist.entropy().mean()
            ratio = torch.exp(logp - mb_old_logp)
            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1.0 - cfg.PPO_CLIP, 1.0 + cfg.PPO_CLIP) * mb_adv
            policy_loss = -torch.mean(torch.min(surr1, surr2))
            value = mappo.critic(mb_g)
            value_loss = torch.mean((mb_ret - value) ** 2)
            action_diversity_loss = 0.0
            if len(move_flat) > 1:
                move_changes = np.sum(move_flat[1:] != move_flat[:-1])
                build_changes = np.sum(build_flat[1:] != build_flat[:-1])
                diversity_ratio = (move_changes + build_changes) / (2 * (len(move_flat) - 1))
                action_diversity_loss = -0.01 * diversity_ratio
            loss = policy_loss + cfg.VALUE_COEF * value_loss - cfg.ENT_COEF * entropy + action_diversity_loss
            mappo.opt_actor.zero_grad()
            mappo.opt_critic.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(mappo.actor.parameters() if mappo.share_actor else mappo.actors[0].parameters(), cfg.MAX_GRAD_NORM)
            nn.utils.clip_grad_norm_(mappo.critic.parameters(), cfg.MAX_GRAD_NORM)
            mappo.opt_actor.step()
            mappo.opt_critic.step()

def save_frames_as_gif(frames, path, fps=6):
    if frames is None or len(frames) == 0:
        raise ValueError("No frames to save (empty list).")
    proc_frames = []
    for i, f in enumerate(frames):
        arr = np.asarray(f)
        if arr.ndim != 3 or arr.shape[2] != 3:
            raise ValueError(f"Frame {i} has wrong shape {arr.shape}, expected (H,W,3).")
        if arr.dtype != np.uint8:
            if np.issubdtype(arr.dtype, np.floating):
                arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
            else:
                arr = arr.astype(np.uint8)
        proc_frames.append(arr)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    imageio.mimsave(path, proc_frames, fps=fps)

def run_eval_episode(env, mappo, deterministic=True, max_steps=500):
    obs_local, global_feat = env.reset()
    frames = []
    steps = 0
    while steps < max_steps:
        obs_arr = np.stack(obs_local, axis=0)
        if deterministic:
            actions_move = []
            actions_build = []
            for i in range(obs_arr.shape[0]):
                ob = torch.tensor(obs_arr[i:i+1], dtype=torch.float32, device=mappo.device)
                actor = mappo.actor if mappo.share_actor else mappo.actors[i]
                with torch.no_grad():
                    move_logits, build_logits = actor(ob)
                    mv = int(torch.argmax(move_logits, dim=1).cpu().item())
                    bd = int(torch.argmax(build_logits, dim=1).cpu().item())
                actions_move.append(mv)
                actions_build.append(bd)
        else:
            mv, bd, _ = mappo.act(obs_arr)
            actions_move = mv.tolist()
            actions_build = bd.tolist()
        obs_tuple, rewards, done, info = env.step(actions_move, actions_build)
        frames.append(env.render_frame())
        steps += 1
        if isinstance(obs_tuple, tuple) and len(obs_tuple) == 2:
            obs_local, global_feat = obs_tuple
        else:
            obs_local = obs_tuple
            global_feat = env.compute_global_features()
        if done:
            break
    if env.runner is not None and env.grid[env.runner] == RUNNER:
        winner = "runner"
    else:
        winner = "catchers"
    return winner, steps, frames

def train_mappo(cfg: Config, num_updates=100, share_actor=True, pretrain_path=None, eval_interval=5, max_eval_gifs=20):
    env = GridEnv(cfg)
    mappo = MAPPO(cfg, share_actor=share_actor)
    if pretrain_path is not None:
        print("Pretraining actor(s) with BC from", pretrain_path)
        pretrain_bc(mappo, demo_path=pretrain_path, epochs=cfg.BC_EPOCHS, batch_size=cfg.BC_BATCH)
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)
    gif_dir = os.path.join(cfg.SAVE_DIR, "eval_gifs")
    os.makedirs(gif_dir, exist_ok=True)
    saved_gifs = 0
    total_episode_count = 0
    for update in range(num_updates):
        obs_flat, global_flat, move_flat, build_flat, reward_flat, done_flat, old_logp_flat, ep_summaries = \
            collect_rollout(env, mappo, cfg.STEPS_PER_UPDATE)
        for s in ep_summaries:
            total_episode_count += 1
            print(f"Episode {total_episode_count}/{num_updates*10} — winner: {s['winner']}, steps: {s['steps']}")
            if total_episode_count <= 5 and saved_gifs < max_eval_gifs:
                gif_path = os.path.join(gif_dir, f"first_ep_{total_episode_count}.gif")
                try:
                    frames = s.get("frames", None)
                    if frames and len(frames) > 0:
                        save_frames_as_gif(frames, gif_path, fps=6)
                        saved_gifs += 1
                        print(f"Saved first GIF: {gif_path}")
                except Exception as e:
                    print("Could not save first GIF:", e)
            if (update + 1) % eval_interval == 0 and saved_gifs < max_eval_gifs:
                gif_path = os.path.join(gif_dir, f"train_ep_{total_episode_count}.gif")
                try:
                    frames = s.get("frames", None)
                    if not frames or len(frames) == 0:
                        winner_eval, steps_eval, frames = run_eval_episode(env, mappo, deterministic=True, max_steps=cfg.MAX_EPISODE_STEPS)
                    save_frames_as_gif(frames, gif_path, fps=6)
                    saved_gifs += 1
                    print(f"Saved GIF: {gif_path}")
                except Exception as e:
                    print("Could not save GIF:", e)
        ppo_update(mappo, obs_flat, global_flat, move_flat, build_flat, reward_flat, done_flat, old_logp_flat, cfg)
        print(f"[Update {update+1}/{num_updates}] collected {obs_flat.shape[0]} samples, mean reward {reward_flat.mean():.4f}")
        if (update + 1) % 10 == 0:
            mappo.save(cfg.SAVE_DIR)
    return mappo

if __name__ == "__main__":
    cfg = Config()
    env = GridEnv(cfg)
    mappo_agent = train_mappo(cfg, num_updates=300, share_actor=True, pretrain_path="demo_data.npz")
    print("Training complete. Checkpoints in", cfg.SAVE_DIR)

---


In [ ]:
#@title Demo generator: Catchers go to assigned exits and build ring (Colab-ready)
import os
import math
import random
import numpy as np
from collections import defaultdict
import pygame
import imageio
import pickle

# Headless pygame for Colab
os.environ["SDL_VIDEODRIVER"] = "dummy"

# -----------------------------
# Config
# -----------------------------
class Config:
    H = 32
    W = 32
    NUM_CATCHERS = 3
    NUM_EXITS = 3
    INTERNAL_WALL_PERCENT = 0.08
    OBS_H = 7
    OBS_W = 7
    RENDER_SCALE = 12
    GIF_FPS = 6

# -----------------------------
# Constants
# -----------------------------
EMPTY = ' '
WALL = 'W'
RUNNER = 'R'
CATCHER = 'C'
EXIT = 'E'

MOVE_DIRS = {
    1: (-1, 0),  # up
    2: (1, 0),   # down
    3: (0, -1),  # left
    4: (0, 1),   # right
}

# -----------------------------
# Simple grid environment (minimal, tailored for demo generation)
# -----------------------------
class CatchersEnv:
    def __init__(self, cfg: Config, seed=None):
        self.cfg = cfg
        self.H = cfg.H
        self.W = cfg.W
        self.n_catchers = cfg.NUM_CATCHERS
        self.n_exits = cfg.NUM_EXITS
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
        self.grid = None
        self.catchers = []   # list of (h,w)
        self.exits = []      # list of (h,w)
        self.runner = None   # optional runner pos
        self.frames = []
        self.reset()

    def reset(self):
        # initialize empty grid
        self.grid = np.full((self.H, self.W), EMPTY, dtype='<U1')
        # frame walls
        self.grid[0, :] = WALL
        self.grid[-1, :] = WALL
        self.grid[:, 0] = WALL
        self.grid[:, -1] = WALL
        # internal random walls
        inner = [(i, j) for i in range(1, self.H-1) for j in range(1, self.W-1)]
        random.shuffle(inner)
        num_walls = int(self.cfg.INTERNAL_WALL_PERCENT * len(inner))
        for i in range(num_walls):
            h, w = inner[i]
            self.grid[h, w] = WALL
        # place exits
        free = [(i, j) for i in range(self.H) for j in range(self.W) if self.grid[i, j] == EMPTY]
        random.shuffle(free)
        self.exits = []
        for _ in range(self.n_exits):
            if not free: break
            h, w = free.pop()
            self.grid[h, w] = EXIT
            self.exits.append((h, w))
        # place runner (optional) - put somewhere not interfering
        free = [(i, j) for i in range(self.H) for j in range(self.W) if self.grid[i, j] == EMPTY]
        random.shuffle(free)
        if free:
            self.runner = free.pop()
            self.grid[self.runner[0], self.runner[1]] = RUNNER
        else:
            self.runner = None
        # place catchers
        free = [(i, j) for i in range(self.H) for j in range(self.W) if self.grid[i, j] == EMPTY]
        random.shuffle(free)
        self.catchers = []
        for _ in range(self.n_catchers):
            if not free: break
            h, w = free.pop()
            self.grid[h, w] = CATCHER
            self.catchers.append((h, w))
        self.frames = []
        return self.get_observation()

    def in_bounds(self, h, w):
        return 0 <= h < self.H and 0 <= w < self.W

    def get_local_window(self, center, oh, ow):
        ch, cw = center
        half_h = oh // 2
        half_w = ow // 2
        top = ch - half_h
        left = cw - half_w
        window = np.full((oh, ow), WALL, dtype='<U1')
        for i in range(oh):
            for j in range(ow):
                gh = top + i
                gw = left + j
                if self.in_bounds(gh, gw):
                    window[i, j] = self.grid[gh, gw]
        return window, (top, left)

    def encode_window_onehot(self, window):
        oh, ow = window.shape
        v = np.zeros((oh, ow, 5), dtype=np.float32)
        for i in range(oh):
            for j in range(ow):
                c = window[i, j]
                if c == EMPTY:
                    v[i, j, 0] = 1.0
                elif c == WALL:
                    v[i, j, 1] = 1.0
                elif c == RUNNER:
                    v[i, j, 2] = 1.0
                elif c == CATCHER:
                    v[i, j, 3] = 1.0
                elif c == EXIT:
                    v[i, j, 4] = 1.0
        return v

    def get_observation(self):
        # return list of local obs for each catcher and simple global vector
        local = []
        for pos in self.catchers:
            w, origin = self.get_local_window(pos, self.cfg.OBS_H, self.cfg.OBS_W)
            local.append(self.encode_window_onehot(w))
        # global features: runner pos normalized, exits mean pos, wall density
        rp = np.array([0.0, 0.0], dtype=np.float32)
        if self.runner is not None:
            rp = np.array([self.runner[0]/self.H, self.runner[1]/self.W], dtype=np.float32)
        if self.exits:
            ex = np.array(self.exits, dtype=np.float32)
            ex_mean = np.mean(ex / np.array([self.H, self.W], dtype=np.float32), axis=0)
            ex_count = len(self.exits) / max(1, self.cfg.NUM_EXITS)
        else:
            ex_mean = np.array([0.0, 0.0], dtype=np.float32)
            ex_count = 0.0
        wall_density = float(np.sum(self.grid == WALL) / (self.H * self.W))
        global_feat = np.concatenate([rp, ex_mean, np.array([ex_count, wall_density], dtype=np.float32)])
        return local, global_feat

    def can_build(self, h, w):
        if not self.in_bounds(h, w): return False
        return self.grid[h, w] == EMPTY

    def step(self, composite_actions):
        """
        composite_actions: list of (move_dir, build_dir) per catcher
         - move_dir: 1..4 or 0 (0 means no movement)
         - build_dir: 1..4 or 0 (0 means no build)
        Execution order: movements for all catchers (in order), then builds (in order).
        But to avoid collisions we will process movement sequentially and prevent moving into occupied cells.
        Returns: (obs_local, global_feat), rewards (zeros), done (False), info
        """
        # Process movements sequentially, preventing collisions
        new_positions = list(self.catchers)
        occupied = set(self.catchers)  # current occupied by catchers
        # First, compute desired moves
        desired = []
        for idx, (mv, bd) in enumerate(composite_actions):
            if mv is None or mv == 0:
                desired.append(self.catchers[idx])
            else:
                dh, dw = MOVE_DIRS.get(mv, (0,0))
                nh = self.catchers[idx][0] + dh
                nw = self.catchers[idx][1] + dw
                # invalid moves (wall, exit, runner, out of bounds) -> stay
                if not self.in_bounds(nh, nw) or self.grid[nh, nw] in (WALL, EXIT, RUNNER):
                    desired.append(self.catchers[idx])
                else:
                    desired.append((nh, nw))
        # Resolve collisions: if multiple want same cell, all those stay
        counts = {}
        for pos in desired:
            counts[pos] = counts.get(pos, 0) + 1
        final_positions = []
        for idx, pos in enumerate(desired):
            if counts[pos] > 1:
                final_positions.append(self.catchers[idx])  # cancel move
            else:
                # also cannot move into cell currently occupied by another catcher who stays
                # check if pos is occupied by a catcher who is not moving away
                occupied_now = set(self.catchers)
                # if target cell is occupied by someone who moves away (desired != their current), allow
                allow = True
                if pos in occupied_now:
                    # find owner
                    owner_idx = None
                    for j, cur in enumerate(self.catchers):
                        if cur == pos:
                            owner_idx = j
                            break
                    if owner_idx is not None and desired[owner_idx] == pos:
                        # owner stays -> cannot move in
                        allow = False
                    elif owner_idx is not None and desired[owner_idx] != pos:
                        # owner moves away -> allow
                        allow = True
                if allow:
                    final_positions.append(pos)
                else:
                    final_positions.append(self.catchers[idx])
        # Update grid: remove old catcher marks
        for (h,w) in self.catchers:
            if self.grid[h,w] == CATCHER:
                self.grid[h,w] = EMPTY
        # Place new positions
        self.catchers = final_positions
        for (h,w) in self.catchers:
            # if cell is empty (should be), mark catcher
            if self.grid[h,w] == EMPTY:
                self.grid[h,w] = CATCHER
            else:
                # if collided with something unexpected, keep as catcher but don't overwrite exit/runner/wall
                pass

        # Process builds sequentially (each build_dir is relative to current catcher position)
        for idx, (mv, bd) in enumerate(composite_actions):
            if bd is None or bd == 0:
                continue
            ch, cw = self.catchers[idx]
            dh, dw = MOVE_DIRS.get(bd, (0,0))
            bh, bw = ch + dh, cw + dw
            if self.in_bounds(bh, bw) and self.can_build(bh, bw):
                # do not build on exits or runner or catchers
                if self.grid[bh, bw] == EMPTY:
                    self.grid[bh, bw] = WALL

        obs, gf = self.get_observation()
        rewards = np.zeros((self.n_catchers,), dtype=np.float32)
        done = False
        info = {}
        return (obs, gf), rewards, done, info

    # -----------------------------
    # Rendering
    # -----------------------------
    def render_to_frame(self):
        scale = self.cfg.RENDER_SCALE
        surf = pygame.Surface((self.W*scale, self.H*scale))
        # colors
        wall_c = (60,60,60)
        empty_c = (240,240,240)
        exit_c = (250,210,0)
        runner_c = (240,50,50)
        catcher_c = (50,90,240)
        for i in range(self.H):
            for j in range(self.W):
                rect = pygame.Rect(j*scale, i*scale, scale, scale)
                c = self.grid[i,j]
                if c == WALL:
                    color = wall_c
                elif c == EXIT:
                    color = exit_c
                elif c == RUNNER:
                    color = runner_c
                elif c == CATCHER:
                    color = catcher_c
                else:
                    color = empty_c
                surf.fill(color, rect)
        # grid lines
        grid_color = (200,200,200)
        for i in range(self.H+1):
            pygame.draw.line(surf, grid_color, (0, i*scale), (self.W*scale, i*scale), 1)
        for j in range(self.W+1):
            pygame.draw.line(surf, grid_color, (j*scale, 0), (j*scale, self.H*scale), 1)
        arr = pygame.surfarray.array3d(surf)
        arr = np.transpose(arr, (1,0,2))
        return arr

    def save_gif_from_frames(self, frames, path):
        if not frames:
            return
        os.makedirs(os.path.dirname(path), exist_ok=True)
        imageio.mimsave(path, frames, fps=self.cfg.GIF_FPS)

# -----------------------------
# Pathfinding (A* Manhattan)
# -----------------------------
def a_star_search(env_state, start, goals):
    H, W = env_state.H, env_state.W
    start = tuple(start)
    goal_set = set(goals)
    if start in goal_set:
        return [start]
    def h(p):
        return min(abs(p[0]-g[0]) + abs(p[1]-g[1]) for g in goals)
    open_set = {start}
    came_from = {}
    gscore = {start: 0}
    fscore = {start: h(start)}
    def lowest():
        return min(open_set, key=lambda p: fscore.get(p, 1e9))
    while open_set:
        current = lowest()
        if current in goal_set:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return list(reversed(path))
        open_set.remove(current)
        ch, cw = current
        for dh, dw in MOVE_DIRS.values():
            nh, nw = ch + dh, cw + dw
            if not (0 <= nh < H and 0 <= nw < W):
                continue
            if env_state.grid[nh, nw] == WALL:
                continue
            neighbor = (nh, nw)
            tentative = gscore[current] + 1
            if tentative < gscore.get(neighbor, 1e9):
                came_from[neighbor] = current
                gscore[neighbor] = tentative
                fscore[neighbor] = tentative + h(neighbor)
                open_set.add(neighbor)
    return []

# -----------------------------
# Scripted episode (each catcher assigned its own exit)
# -----------------------------
def dir_to_target(cx, cy, tx, ty):
    # mapping to env MOVE_DIRS keys (1..4)
    if tx < cx: return 1
    if tx > cx: return 2
    if ty < cy: return 3
    if ty > cy: return 4
    return 0

def step_from_dir(cx, cy, move_dir):
    if move_dir == 1: return (cx-1, cy)
    if move_dir == 2: return (cx+1, cy)
    if move_dir == 3: return (cx, cy-1)
    if move_dir == 4: return (cx, cy+1)
    return (cx, cy)

def scripted_episode(env: CatchersEnv, max_steps=300, verbose=False):
    obs = env.reset()
    frames = []
    episode_data = []

    exits = list(env.exits)
    assigned_exits = {}
    used_exits = set()

    for step in range(max_steps):
        composite_actions = []

        # assign exits to catchers (closest free)
        for idx, (cx, cy) in enumerate(env.catchers):
            if idx not in assigned_exits:
                sorted_exits = sorted(exits, key=lambda e: abs(e[0]-cx)+abs(e[1]-cy))
                for ex in sorted_exits:
                    if ex not in used_exits:
                        assigned_exits[idx] = ex
                        used_exits.add(ex)
                        break
                if idx not in assigned_exits and sorted_exits:
                    assigned_exits[idx] = sorted_exits[0]

        # plan actions
        for idx, (cx, cy) in enumerate(env.catchers):
            ex = assigned_exits.get(idx, None)
            move_dir = 0
            build_dir = 0
            if ex is None:
                # no assigned exit -> random move
                mv = random.choice([1,2,3,4])
                move_dir = mv
                composite_actions.append((move_dir, build_dir))
                continue

            # ring cells (4-neighbors) around exit
            ring = [(ex[0]-1, ex[1]), (ex[0]+1, ex[1]), (ex[0], ex[1]-1), (ex[0], ex[1]+1)]
            ring_cells = [(rx, ry) for (rx, ry) in ring if env.in_bounds(rx, ry) and env.grid[rx, ry] == EMPTY]

            if ring_cells:
                # choose nearest ring cell
                target = min(ring_cells, key=lambda p: abs(p[0]-cx)+abs(p[1]-cy))
                # compute path to target (A*). We want to stop one cell before target (i.e., be adjacent)
                path = a_star_search(env, (cx, cy), [target])
                if path and len(path) >= 2:
                    # path[0] == start, path[-1] == target
                    # stop_cell is the cell before target
                    stop_cell = path[-2] if len(path) >= 2 else path[0]
                    # if currently at stop_cell -> build into target
                    if (cx, cy) == stop_cell:
                        tx, ty = target
                        # determine build_dir relative to catcher
                        if tx == cx-1 and ty == cy: build_dir = 1
                        elif tx == cx+1 and ty == cy: build_dir = 2
                        elif tx == cx and ty == cy-1: build_dir = 3
                        elif tx == cx and ty == cy+1: build_dir = 4
                        # movement is zero this step (we perform build)
                        move_dir = 0
                    else:
                        # move one step along path (first step after start)
                        next_step = path[1]
                        move_dir = dir_to_target(cx, cy, next_step[0], next_step[1])
                        build_dir = 0
                else:
                    # no path to target (rare) -> random move
                    move_dir = random.choice([1,2,3,4])
                    build_dir = 0
            else:
                # nothing to build around this exit -> idle or random explore
                move_dir = random.choice([1,2,3,4])
                build_dir = 0

            # if catcher accidentally stands on ring cell (adjacent to exit) but not in stop_cell logic,
            # ensure it doesn't block: if adjacent and not building, try to step away
            if abs(cx - ex[0]) + abs(cy - ex[1]) == 1 and build_dir == 0:
                # step away from exit if possible
                moved = False
                for d in [1,2,3,4]:
                    nx, ny = step_from_dir(cx, cy, d)
                    if env.in_bounds(nx, ny) and env.grid[nx, ny] == EMPTY and abs(nx-ex[0]) + abs(ny-ex[1]) > 1:
                        move_dir = d
                        build_dir = 0
                        moved = True
                        break
                if not moved:
                    # stay and try to build if target empty
                    # find which neighbor is empty and build into it (but only if it's ring cell)
                    for (tx, ty) in ring:
                        if env.in_bounds(tx, ty) and env.grid[tx, ty] == EMPTY:
                            if tx == cx-1 and ty == cy: build_dir = 1
                            elif tx == cx+1 and ty == cy: build_dir = 2
                            elif tx == cx and ty == cy-1: build_dir = 3
                            elif tx == cx and ty == cy+1: build_dir = 4
                            move_dir = 0
                            break

            composite_actions.append((move_dir, build_dir))

        # collision resolution for moves: if multiple want same cell -> cancel moves for those
        desired_positions = []
        for idx, (cx, cy) in enumerate(env.catchers):
            mv, bd = composite_actions[idx]
            if mv == 0:
                desired_positions.append((cx, cy))
            else:
                desired_positions.append(step_from_dir(cx, cy, mv))
        counts = {}
        for pos in desired_positions:
            counts[pos] = counts.get(pos, 0) + 1
        # cancel moves where conflict
        for idx, pos in enumerate(desired_positions):
            if counts[pos] > 1:
                mv, bd = composite_actions[idx]
                composite_actions[idx] = (0, bd)

        # apply step
        (obs_next, gf), rewards, done, info = env.step(composite_actions)

        # record step
        episode_data.append({
            "obs": obs,               # full obs (local list, global vector)
            "actions": composite_actions,
            "rewards": rewards.tolist(),
            "done": done,
            "info": info
        })
        frames.append(env.render_to_frame())
        obs = (obs_next, gf)

    return episode_data, frames

# -----------------------------
# Generator: produce many episodes, save .npz and gifs
# -----------------------------
def generate_and_save(num_episodes=20, max_steps=200, out_npz="demo_data.npz", gif_dir="gifs"):
    env = CatchersEnv(Config(), seed=42)
    os.makedirs(gif_dir, exist_ok=True)

    all_obs_local = []
    all_global = []
    all_moves = []
    all_builds = []
    all_rewards = []
    all_dones = []

    for ep in range(num_episodes):
        ep_data, frames = scripted_episode(env, max_steps=max_steps, verbose=False)
        # save gif
        gif_path = os.path.join(gif_dir, f"demo_ep_{ep+1}.gif")
        try:
            env.save_gif_from_frames(frames, gif_path)
        except Exception:
            pass

        # flatten steps
        for step_entry in ep_data:
            obs_step = step_entry["obs"]
            # obs_step is (local_list, global_vec) as returned by env.reset/get_observation
            if isinstance(obs_step, tuple) and len(obs_step) == 2:
                local_list, global_vec = obs_step
            else:
                local_list, global_vec = None, None
            all_obs_local.append(local_list)
            all_global.append(global_vec)
            actions = step_entry["actions"]
            moves = np.array([a[0] for a in actions], dtype=np.int32)
            builds = np.array([a[1] for a in actions], dtype=np.int32)
            all_moves.append(moves)
            all_builds.append(builds)
            all_rewards.append(np.array(step_entry["rewards"], dtype=np.float32))
            all_dones.append(bool(step_entry["done"]))

        print(f"Episode {ep+1}/{num_episodes} saved, steps={len(ep_data)}")

    # convert to arrays
    obs_local_arr = np.array(all_obs_local, dtype=object)
    global_arr = np.array(all_global, dtype=object)
    moves_arr = np.stack(all_moves, axis=0)   # (T, n_catchers)
    builds_arr = np.stack(all_builds, axis=0)
    rewards_arr = np.stack(all_rewards, axis=0)
    dones_arr = np.array(all_dones, dtype=np.bool_)

    np.savez_compressed(out_npz,
                        obs_local=obs_local_arr,
                        global_feat=global_arr,
                        actions_move=moves_arr,
                        actions_build=builds_arr,
                        rewards=rewards_arr,
                        done=dones_arr)
    print(f"Saved demo data to {out_npz} and GIFs to {gif_dir}")

# -----------------------------
# Run generator (example)
# -----------------------------
if __name__ == "__main__":
    # generate 12 episodes by default
    generate_and_save(num_episodes=200, max_steps=100, out_npz="demo_data.npz", gif_dir="gifs")
    print("Done. Files: demo_data.npz and gifs/*")
